# CDC + Earthquake GAS Workflow

This notebook demonstrates a GAS SDK workflow focused on:

1. Data downloading with `geospatial_data_retrieval_agent`.
2. Vector analysis with `vector_analysis_agent`.
3. Static cartographic output with `mapping_agent`.
4. Web mapping app development with `web_mapping_app_agent`.

All service calls use `mode="stream"` so progress messages are visible while each agent works.

The workflow uses two no-key public data sources:

- **CDC PLACES** for county-level health data.
- **USGS Earthquake** for recent earthquake event data.

## Install GAS Client SDK

This notebook uses the published `gas-client` package from PyPI. Run this cell once in a new notebook environment.


In [ ]:
%pip install -q gas-client


## Imports

In [ ]:
import os
from pathlib import Path
from urllib.parse import urljoin

from IPython.display import HTML, Image, display

project_root = Path.cwd()
if project_root.name == "examples_for_using_gas_services":
    project_root = project_root.parent


from gas_client import GasClient

## User Settings

Start the GAS server first. For local development, keep `server_url` as `http://127.0.0.1:4042`.

CDC PLACES and USGS Earthquake do not require source API keys. The OpenAI key is still needed because these agents use model-backed reasoning.

In [ ]:
server_url = "http://127.0.0.1:4042"
openai_api_key = "YOUR_OPENAI_API_KEY"
timeout_seconds = 900

client = GasClient(
    server_url,
    openai_api_key=openai_api_key,
    artifact_delivery="URL",
)

credentials = {
    "OPENAI_API_KEY": openai_api_key,
}

## Discover Agents

In [ ]:
client.list_agents()

In [ ]:
data_agent = client.agent("geospatial_data_retrieval_agent")
vector_agent = client.agent("vector_analysis_agent")
mapping_agent = client.agent("mapping_agent")
web_mapping_app_agent = client.agent("web_mapping_app_agent")

for agent in [data_agent, vector_agent, mapping_agent, web_mapping_app_agent]:
    print(agent.agent_id, agent.status().get("status"))

## Helper Functions

`run_streaming_task()` calls one GAS agent in streaming mode, prints stream events, and returns the final standard GAS task response.

In [ ]:
def absolute_url(url):
    if url.startswith("/"):
        return urljoin(server_url, url)
    return url


def first_artifact_url(task_result, preferred_extensions=None):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    preferred_extensions = preferred_extensions or []

    for extension in preferred_extensions:
        for artifact in artifacts:
            url = artifact.get("url")
            filename = artifact.get("filename") or artifact.get("name") or url or ""
            if url and str(filename).lower().endswith(extension.lower()):
                return absolute_url(url)

    for artifact in artifacts:
        if artifact.get("url"):
            return absolute_url(artifact["url"])

    raise RuntimeError("The task returned no artifact URL.")


def display_visual_artifacts(task_result):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    for artifact in artifacts:
        url = artifact.get("url")
        filename = artifact.get("filename") or artifact.get("name") or ""
        if not url:
            continue

        display_url = absolute_url(url)
        lower_name = str(filename).lower()
        if lower_name.endswith(".png"):
            display(Image(url=display_url))
        elif lower_name.endswith(".html"):
            display(HTML(f'<a href="{display_url}" target="_blank">Open interactive HTML artifact: {filename}</a>'))
        else:
            display(HTML(f'<a href="{display_url}" target="_blank">Open artifact: {filename}</a>'))


def run_streaming_task(agent, instructions, *, input_datasets=None, parameters=None, title=None):
    if title:
        print("\n" + "=" * 80)
        print(title)
        print("=" * 80)

    final_result = None
    for event in agent.execute_task(
        instructions,
        mode="stream",
        input_datasets=input_datasets,
        artifact_delivery="URL",
        credentials=credentials,
        parameters=parameters,
        timeout=timeout_seconds,
    ):
        client.print_stream_event(event)
        if event.get("event") == "task_result":
            final_result = event.get("payload")

    if final_result is None:
        raise RuntimeError("The stream ended before returning a task_result event.")

    client.print_task_summary(final_result)
    return final_result

## Part 1: Download Contiguous US County Boundaries

First, use the data retrieval agent to download the county boundary geometry for the contiguous United States. This becomes the geometry layer for the later join and choropleth map.

In [ ]:
county_boundary_result = run_streaming_task(
    data_agent,
    (
        "Use the US Census Bureau boundary source. Download county boundaries for the contiguous United States. "
        "Exclude Alaska, Hawaii, Puerto Rico, and other territories. Return one clean GeoPackage dataset with "
        "county geometry, county name, state name or state abbreviation, STATEFP, COUNTYFP, and GEOID fields. "
        "No source API key is required."
    ),
    title="Download Contiguous US County Boundaries",
)

county_boundary_url = first_artifact_url(county_boundary_result, preferred_extensions=[".gpkg", ".geojson", ".json"])
county_boundary_url

## Part 2: Download 2020 County-Level Obesity Rates from CDC PLACES

Next, use the data retrieval agent again to download an attribute table or geospatial dataset containing county-level adult obesity prevalence for 2021. The vector analysis agent will join this to the county boundaries.

In [ ]:
obesity_result = run_streaming_task(
    data_agent,
    (
        "Download 2020 county-level adult obesity prevalence estimates "
        "for the contiguous United States. Return one clean dataset with county FIPS/GEOID, county name, "
        "state identifier, year, measure name, and the obesity prevalence value. Geometry is optional for this "
        "dataset because it will be joined to a separate county boundary dataset. No source API key is required."
    ),
    title="Download 2020 County-Level Obesity Rates from CDC PLACES",
)

obesity_url = first_artifact_url(obesity_result, preferred_extensions=[".gpkg", ".csv", ".geojson", ".json"])
obesity_url

## Part 3: Join County Boundaries with Obesity Rates

The vector analysis agent receives both datasets and creates one mapping-ready GeoJSON. The join should use county GEOID/FIPS fields and preserve the county boundary geometry.

In [ ]:
obesity_join_result = run_streaming_task(
    vector_agent,
    (
        "Join the contiguous US county boundary dataset with the 2020 county-level adult obesity prevalence dataset. "
        "Use GEOID or county FIPS fields as the join key. Preserve county boundary geometry and key identifiers. "
        "Create a clean numeric field named obesity_rate for the obesity prevalence value. Keep county name, state, GEOID, "
        "year, and measure fields when available. Exclude unmatched non-contiguous records if present. Return one GeoJSON "
        "artifact ready for choropleth mapping."
    ),
    input_datasets=[county_boundary_url, obesity_url],
    title="Join County Boundaries and CDC Obesity Rates",
)

obesity_join_url = first_artifact_url(obesity_join_result, preferred_extensions=[".geojson", ".gpkg", ".json"])
obesity_join_url

## Part 4: Static Choropleth Map of County Obesity Rates

The static `mapping_agent` creates a presentation-ready PNG choropleth map from the joined county geometry and obesity attribute dataset.

In [ ]:
obesity_map_result = run_streaming_task(
    mapping_agent,
    (
        "Create a professional county-level choropleth map for the contiguous United States showing 2020 adult obesity rate. "
        "Use the obesity_rate field for color. Use 5 quantile classes, a clear sequential color scheme, thin county outlines, "
        "missing values in light gray, a legend outside the map axes, and the title 'County-Level Adult Obesity Rate, 2020'." \
        "Use the Albers Equal Area projection. " \
    ),
    input_datasets=[obesity_join_url],
    title="Create Static Obesity Rate Choropleth Map",
)

display_visual_artifacts(obesity_map_result)


## Part 5: Download USGS Earthquake Data

This request uses the USGS Earthquake source. The output will be used by the web mapping app agent.

In [ ]:
earthquake_result = run_streaming_task(
    data_agent,
    (
        "Use the USGS Earthquake data source. Download recent earthquakes from the last 30 days "
        "with magnitude 2.5 or greater for California and nearby areas. Return one GeoJSON point "
        "dataset with event time, magnitude, place/location description, depth, event id, and geometry. "
        "No source API key is required."
    ),
    title="Download USGS Earthquake Events",
)

earthquake_url = first_artifact_url(earthquake_result, preferred_extensions=[".geojson", ".json", ".gpkg"])
earthquake_url


## Part 6: Interactive Earthquake Map

The web mapping app agent creates a browser-ready HTML web mapping app from the earthquake points. It should include a map, layer control, popups, a legend, a professional title, and app-style layout where useful.

In [ ]:
earthquake_map_result = run_streaming_task(
    web_mapping_app_agent,
    (
        "Create a polished web mapping app of the earthquake events. Use circle markers sized by magnitude "
        "and colored by magnitude class. Include popups with event time, magnitude, depth, place, and event id. "
        "Include a layer control, a visible legend, a professional title, and an initial extent that fits all events. "
        "Use a clean basemap suitable for earthquake visualization."
    ),
    input_datasets=[earthquake_url],
    title="Create Interactive USGS Earthquake Map",
)

display_visual_artifacts(earthquake_map_result)


## Optional: Inspect Final Artifact URLs

In [ ]:
workflow_outputs = {
    "county_boundaries": county_boundary_url,
    "cdc_obesity_rates": obesity_url,
    "joined_obesity_counties": obesity_join_url,
    "earthquake_download": earthquake_url,
}

workflow_outputs
